# 05 - Writing Small Outputs

IceTray `.i3` files are the native IceCube event format. They are not always the most convenient format for quick analysis/plotting.

This notebook shows two common output patterns:

1. Use `I3HDFWriter` to write selected frame keys to HDF5.
2. Build a small Pandas table yourself and save it to HDF5.


In [1]:
from pathlib import Path
import os

import pandas as pd
from icecube import dataio, dataclasses
from I3Tray import I3Tray
from icecube.hdfwriter import I3HDFWriter

DATA_GCD = Path('/data/exp/IceCube/2020/filtered/level2/0101/Run00133576/Level2_IC86.2019_data_Run00133576_0101_78_503_GCD.i3.zst')
DATA_FILE = Path('/data/exp/IceCube/2020/filtered/level2/0101/Run00133576/Level2_IC86.2019_data_Run00133576_Subrun00000000_00000000.i3.zst')

username = os.environ.get('USER', 'icecube_user')
output_dir = Path(f'/data/user/{username}/IceTray_tutorial')
output_dir.mkdir(parents=True, exist_ok=True)

print('Data file exists:', DATA_FILE.exists())
print('Output directory:', output_dir)


Data file exists: True
Output directory: /data/user/ireistr/IceTray_tutorial


/tmp/ipykernel_8724/365832442.py:6: UserWarning: Using `import I3Tray` or `from I3Tray import *` is now considered deprecated. Please switch to using `from icecube.icetray import I3Tray`
  from I3Tray import I3Tray


## Option A: I3HDFWriter

`I3HDFWriter` is an IceTray tool. It reads frames from a tray and writes selected frame keys to an HDF5 file.

Here we start with the objects `I3EventHeader` and `QFilterMask`.


In [2]:
hdf_output = output_dir / 'selected_keys_from_i3hdfwriter.hdf5'

tray = I3Tray()
tray.AddModule('I3Reader', 'reader', FilenameList=[str(DATA_GCD), str(DATA_FILE)])
tray.AddSegment(
    I3HDFWriter,
    'hdf_writer',
    Output=str(hdf_output),
    Keys=['I3EventHeader', 'QFilterMask'],
    SubEventStreams=['InIceSplit'],
)

print('Writing a small HDF5 file with I3HDFWriter...')
tray.Execute(500)
tray.Finish()
print('Wrote:', hdf_output)


Writing a small HDF5 file with I3HDFWriter...
Wrote: /data/user/ireistr/IceTray_tutorial/selected_keys_from_i3hdfwriter.hdf5


NOTICE (I3Tray): I3Tray finishing... (I3Tray.cxx:525 in void I3Tray::Execute(bool, unsigned int))
WARN (I3TableWriter): 3 SubEventStreams ['IceTopSplit','NullSplit','SLOPSplit',] were seen but not booked because they were not passed as part of the 'SubEventStreams' parameter of I3TableWriter (which was configured as ['InIceSplit',]). To book events from these streams, add them to the 'SubEventStreams' parameter of I3TableWriter. (I3TableWriter.cxx:473 in void I3TableWriter::Finish())


## Option B: Make your own flat table

It's sometimes easier to create a table with one row per event.

This will contain only the values we choose, not everything in the .i3 frames.

The functions below repeat ideas from earlier notebooks: find P-frames, find a pulse map, count DOMs, add up charge.


In [3]:
def frame_stop(frame):
    names = {'Q': 'DAQ', 'P': 'Physics', 'G': 'Geometry', 'C': 'Calibration', 'D': 'DetectorStatus', 'I': 'TrayInfo'}
    return names.get(str(frame.Stop), str(frame.Stop))

def find_pulse_key(frame):
    for key in ['SplitInIcePulses', 'SplitInIceDSTPulses', 'SRTInIcePulses', 'InIcePulses', 'OfflinePulses']:
        if key in frame:
            return key
    return None

def pulse_summary(frame):
    pulse_key = find_pulse_key(frame)
    if pulse_key is None:
        return {'pulse_key': None, 'hit_doms': None, 'number_of_pulses': None, 'total_charge': None}

    pulse_map = dataclasses.I3RecoPulseSeriesMap.from_frame(frame, pulse_key)
    hit_doms = 0
    number_of_pulses = 0
    total_charge = 0.0

    for omkey, pulses_on_one_dom in pulse_map:
        hit_doms += 1
        number_of_pulses += len(pulses_on_one_dom)
        for pulse in pulses_on_one_dom:
            total_charge += pulse.charge

    return {'pulse_key': pulse_key, 'hit_doms': hit_doms, 'number_of_pulses': number_of_pulses, 'total_charge': total_charge}


## Loop through events and collect rows

Each row is a dictionary.

The dictionary keys become table column names.


In [4]:
rows = []
physics_seen = 0
MAX_PHYSICS_FRAMES = 1000

i3_file = dataio.I3File(str(DATA_FILE))
while i3_file.more() and physics_seen < MAX_PHYSICS_FRAMES:
    frame = i3_file.pop_frame()
    if frame_stop(frame) != 'Physics':
        continue

    physics_seen += 1
    row = {}

    if 'I3EventHeader' in frame:
        header = frame['I3EventHeader']
        row['run_id'] = header.run_id
        row['event_id'] = header.event_id
        row['sub_event_id'] = header.sub_event_id
        row['sub_event_stream'] = str(header.sub_event_stream)

    row.update(pulse_summary(frame))
    rows.append(row)

i3_file.close()

event_table = pd.DataFrame(rows)
print(f'Built a table with {len(event_table)} rows.')
print('Columns:', list(event_table.columns))
event_table.head()


Built a table with 1000 rows.
Columns: ['run_id', 'event_id', 'sub_event_id', 'sub_event_stream', 'pulse_key', 'hit_doms', 'number_of_pulses', 'total_charge']


,run_id,event_id,sub_event_id,sub_event_stream,pulse_key,hit_doms,number_of_pulses,total_charge
0,133576,2061,0,NullSplit,InIcePulses,56,75,71.517496
1,133576,2061,0,InIceSplit,SplitInIcePulses,56,75,71.517496
2,133576,2062,0,NullSplit,InIcePulses,58,63,54.175000
3,133576,2062,0,InIceSplit,SplitInIcePulses,57,62,53.300000
4,133576,2065,0,IceTopSplit,InIcePulses,221,548,493.117124


## Save and read back the table

Pandas can write a DataFrame to HDF5 with `to_hdf`. (Sanity check.)


In [5]:
custom_output = output_dir / 'event_hit_summary_from_pandas.h5'
event_table.to_hdf(custom_output, key='events', mode='w')

print('Saved table to:', custom_output)
print('Reading it back now:')
pd.read_hdf(custom_output, key='events').head()


Saved table to: /data/user/ireistr/IceTray_tutorial/event_hit_summary_from_pandas.h5
Reading it back now:


,run_id,event_id,sub_event_id,sub_event_stream,pulse_key,hit_doms,number_of_pulses,total_charge
0,133576,2061,0,NullSplit,InIcePulses,56,75,71.517496
1,133576,2061,0,InIceSplit,SplitInIcePulses,56,75,71.517496
2,133576,2062,0,NullSplit,InIcePulses,58,63,54.175000
3,133576,2062,0,InIceSplit,SplitInIcePulses,57,62,53.300000
4,133576,2065,0,IceTopSplit,InIcePulses,221,548,493.117124


## Practice

1. Add a column for reconstructed zenith if `SplineMPE` exists in the frame.
2. Add a column that says whether `MuonFilter_13` passed.
3. Change `MAX_PHYSICS_FRAMES` and compare file size.
4. Open the HDF5 file with Pandas in a new notebook and make a histogram from it.
